In [1]:
import os
import sys
from joblib import Parallel, delayed
from tqdm import tqdm
import json
import math

root_dir = os.path.dirname(os.getcwd())
sys.path.append(root_dir)
import _3DMorph.utils.voxel_utils as utils
from _3DMorph.renderer.render_simple import BlenderRenderer
from _3DMorph.slat_encoder.latent_encoder import LatentEncoder
from _3DMorph.slat_encoder.feature_extractor import FeatureExtractor

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
[SPARSE] Backend: spconv, Attention: xformers


No CUDA runtime is found, using CUDA_HOME='/usr/local/cuda'


Warp 1.6.2 initialized:
   CUDA devices not available
   Devices:
     "cpu"      : "x86_64"
   Kernel cache:
     /home/ubuntu/.cache/warp/1.6.2


Warp CUDA error 100: no CUDA-capable device is detected (in function init_cuda_driver, /builds/omniverse/warp/warp/native/cuda_util.cpp:262)
Warp CUDA error 3: initialization error (in function cuda_device_get_count, /builds/omniverse/warp/warp/native/warp.cu:1755)


## 1. Prepare all object folder paths:

In [4]:
path_to_dataset = os.path.join(root_dir, "assets", "Assembly_Pairs")  # Change if needed

dataset_dict = {}

for dataset_part in os.listdir(path_to_dataset):
    if not os.path.isdir(os.path.join(path_to_dataset, dataset_part)):
        continue

    for object_id in os.listdir(os.path.join(path_to_dataset, dataset_part)):
        object_dir = os.path.join(path_to_dataset, dataset_part, object_id)
        if not os.path.isdir(object_dir):
            continue

        # Check whether the original object exists:
        path_to_original = os.path.join(object_dir, "original_assembly.obj")
        assert os.path.exists(path_to_original), f"Path to original object does not exist for {object_id} in {dataset_part}"

        # Check whether the modified object exists:
        path_to_modified_candidates = [x for x in os.listdir(object_dir) if x.endswith(".obj") and "original_assembly" not in x]
        assert len(path_to_modified_candidates) == 1 and os.path.exists(os.path.join(object_dir, path_to_modified_candidates[0])), (
            f"Path to modified object does not exist for {object_id} in {dataset_part} {path_to_modified_candidates}"
        )
        path_to_modified = path_to_modified_candidates[0]

        # Check whether the transforms exists:
        path_to_transforms = os.path.join(object_dir, "transforms.json")
        assert os.path.exists(path_to_transforms), f"Path to transforms does not exist for {object_id} in {dataset_part}"

        dataset_dict[object_id] = {
            "root": object_dir,
            "original": path_to_original,
            "modified": path_to_modified,
            "transforms": path_to_transforms,
        }

print(f"Found {len(dataset_dict)} different objects!")


Found 74 different objects!


## 2. Voxelize the objects

In [ ]:
voxelize_tasks = []

for object_id, data_dict in dataset_dict.items():
    voxelize_tasks += utils.process_directory(data_dict["root"])

Parallel(n_jobs=-1, verbose=10)(
    delayed(utils.process_mesh)(*task) for task in tqdm(
        voxelize_tasks,
        desc="Voxelizing .obj files...",
        total=len(voxelize_tasks),
        unit="file"
    )
)

## 3. Render input views

In [ ]:
renderer = BlenderRenderer(1024)
render_tasks = []

for object_id, data_dict in dataset_dict.items():
    # Load transforms
    with open(data_dict["transforms"], 'r') as f:
        transforms: list[dict] = json.load(f)
    assert len(transforms) > 0, f"No transforms in {data_dict['transforms']}"

    # Extract field-of-view
    fovs = [math.degrees(x["camera_angle_x"]) for x in transforms]
    assert all(fov == fovs[0] for fov in fovs), f"FoVs are not identical for {object_id}"
    fov = fovs[0]

    # Extract initial azimuth, elevation
    azimuth = transforms[0]["yaw"]
    elevation = transforms[0]["pitch"]

    # Determine step size and direction
    if len(transforms) == 1:
        step = 0
        direction = "right"
    else:
        steps =[transforms[i+1]["yaw"] - transforms[i]["yaw"] for i in range(len(transforms) - 1)]
        assert all(step - steps[0] <= 1e-3 for step in steps), f"Steps are not identical for {object_id} {steps}"
        step = steps[0]
        direction = "right" if step >= 0 else "left"
        
    view_args = {'offset': (azimuth, elevation), 'step': step, 'direction': direction, 'set_fov': fov}
    save_dir = os.path.join(data_dict["root"], "input_views")
    os.makedirs(save_dir, exist_ok=True)

    render_tasks.append((data_dict["modified"], {
        "ref_obj_path": data_dict["original"],
        "save_dir": save_dir,
        "n_views": len(transforms),
        "mode": "linear",
        "lin_view_args": view_args,
    }))


def render_view(mesh_path, args):
    _ = renderer.render(
        mesh_path, 
        **args,
    )

Parallel(n_jobs=-1, verbose=10)(
    delayed(render_view)(*task) for task in tqdm(
        render_tasks,
        desc="Rendering input views...",
        total=len(render_tasks),
        unit="file"
    )
)

## 4. Encode each new mesh as <span style="font-size: 32px; font-weight: 600;">SL</span><span style="font-size: 28px; font-weight: 700;">AT</span>

In [ ]:
extractor = FeatureExtractor(batch_size=150, n_views=150)
slat_encoder = LatentEncoder()
slat_tasks = []

for object_id, data_dict in dataset_dict.items():
    slat_tasks.append((data_dict["modified"], data_dict["original"]))


def compute_slat(obj_path, ref_path):
    feat_path = extractor.run_extractor(obj_path, ref_path, force_render=True)
    _ = slat_encoder.run_slat_encoder(feat_path)

Parallel(n_jobs=1, verbose=10)(
    delayed(compute_slat)(*task) for task in tqdm(
        slat_tasks,
        desc="Computing SLats...",
        total=len(slat_tasks),
        unit="file"
    )
)